# AR Model Notebook

1. **Setup** - Import packages and write function
2. **Simulation 1** - Generating synthetic AR(1) data and checking stationarity
3. **Model 1: AR(1) on Synthetic Data** - Basically taken line-for-line from the Stan documentation
4. **Simulation 2** - Generating 5 series of AR(1) data that is generated from a CAR model as described in the Stan documentation.
5. **Model 2: Hierarchical AR (ICAR Prior) on Synthetic Data** - Fitting `ar_1_hierarchical_icar.stan`
6. **Load CTA Data for Stan** - Loading in data, adjacency matrix, etc.
7. **Model 3: Hierarchical AR (ICAR Prior) on CTA Data** - Fitting `ar_1_hierarchical_icar.stan`
8. **Model 4: Hierarchical AR (ICAR Prior) on CTA Data with Seasonality** - Fitting `ar_1_hier_icar_seas.stan`
9. **Posterior Predictive Checks** - Checking whether the estimated DGP fits the actual data

---
## 1. Setup

In [ ]:
# Import Python packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# Import Stan
from cmdstanpy import CmdStanModel, set_cmdstan_path
from cmdstanpy import from_csv
set_cmdstan_path("/deac/sta/classes/sta720/software/cmdstan/2.37.0")

In [ ]:
# load station look-up data frame
station_lookup = pd.read_csv("../../data/clean/station-lookup.csv")

# Function to look-up stations
def lookup_station(value, station_lookup):
    """Look up a CTA station by ID or name. Accepts a scalar or iterable."""
    id_to_name = dict(zip(station_lookup["station_id"], station_lookup["stationname"]))
    name_to_id = dict(zip(station_lookup["stationname"], station_lookup["station_id"]))
    
    def lookup_one(v):
        if isinstance(v, (int, np.integer)):
            return id_to_name.get(v)
        return name_to_id.get(v)
    
    if isinstance(value, (int, str, np.integer)):
        return lookup_one(value)
    
    return [lookup_one(v) for v in value]

In [ ]:
# Function to simulate AR(1) univariate data
def sim_ar_model(alpha, beta, sigma, T, initial=0, seed=None):
    np.random.seed(seed)
    assert T >= 1

    Y = np.zeros(T)
    epsilon = np.random.normal(0, sigma, size=T)

    Y[0] = initial

    for t in range(1, T):
        Y[t] = alpha + beta * Y[t - 1] + epsilon[t]
    return Y

## 2. Simulate AR(1) Data

In [ ]:
# Simulate data and plot
T = 100
alpha = 0
beta= 0.98
sigma = 20

y = sim_ar_model(alpha = alpha, beta = beta, sigma = 20, T = T, seed = 5)

df_sim = pd.DataFrame(y)
df_sim["t"] = range(T)
df_sim_long = df_sim.melt(id_vars="t", var_name="series", value_name="value")

plt.figure(figsize=(10, 4))
sns.lineplot(data=df_sim_long, x="t", y="value", hue="series")
plt.title("Synthetic AR(1) Simulation")
plt.xlabel("Time")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

## 3. Model 1: AR(1) on Simulated Data

In [ ]:
# Run simple stan model (from documentation)
stan_data_ar1_sim = {
    "T": T,
    "y": y
}

ar_mod = CmdStanModel(stan_file="ar_1.stan")
ar_mod_post = ar_mod.sample(data=stan_data_ar1_sim)
print(ar_mod_post.diagnose())

### 3.1 Parameter Recovery

In [ ]:
# Recover parameters
ar_mod_params = ar_mod_post.stan_variables()

print("True alpha:", alpha)
print("Estimated alpha:", ar_mod_params["alpha"].mean(axis=0))
print()
print("True sigma:", sigma)
print("Estimated sigma:", ar_mod_params["sigma"].mean(axis=0))
print()
print("True sigma:", sigma)
print("Estimated sigma:", ar_mod_params["sigma"].mean(axis=0))

In [ ]:
print(ar_mod_post.summary())

## 4. Simulate Network AR(1) Data

In [ ]:
# Simulate independent AR(1) data function
def sim_ar_net_model(c, beta, T, initial=None, seed=None):
    if seed is not None:
        np.random.seed(seed)
    K = len(c)
    Y = np.zeros((T, K))
    if initial is not None:
        Y[0] = initial
    else:
        Y[0] = c / (1 - beta) # Start at stationary mean
    
    for t in range(1, T):
        epsilon = np.random.normal(0, 1, K)  # independent innovations
        Y[t] = c + beta * Y[t-1] + epsilon
    
    return Y


In [ ]:
# Simulation parameters
K = 5
c_net = np.array([0.7, 0.05, 0.6, 0.12, 0.5])       # all node-level intercepts
beta_net = np.array([-0.45, -0.40, -0.30, -0.25, -0.18])   # all node-level coefficients
gamma_net = c_net.mean()                             # population intercept (hierarchical mean)
delta_net = beta_net.mean()                          # population AR coefficient (hierarchical mean)
T_net = 200

Y_net = sim_ar_net_model(c=c_net, beta=beta_net, T=T_net, seed=42)

print(gamma_net)
print(delta_net)

In [ ]:
# Plot all 5 nodes
df_net = pd.DataFrame(Y_net, columns=[f"Node {i+1}" for i in range(K)])
df_net["t"] = range(T_net)
df_net_long = df_net.melt(id_vars="t", var_name="node", value_name="value")

plt.figure(figsize=(12, 5))
sns.lineplot(data=df_net_long, x="t", y="value", hue="node")
plt.title("Network AR(1)")
plt.xlabel("Time")
plt.ylabel("Value")
plt.tight_layout()
plt.show()

## 5. Model 2: Hierarchical AR(1) with ICAR Prior on Simulated Network Data

In [ ]:
node1 = []
node2 = []

# Set-up adjacency matrix
W = np.zeros((K, K))
for i in range(K - 1):
    W[i, i + 1] = 1
    W[i + 1, i] = 1

# Split up adjacency matrix into vectors
for i in range(K):
    for j in range(i + 1, K):
        if W[i, j] == 1:
            node1.append(i + 1)
            node2.append(j + 1)

print(node1)
print(node2)

# stan data
stan_data_icar = {
    "T": T_net,
    "K": K,
    "Y": Y_net,
    "N_edges": N_edges,
    "node1": node1,
    "node2": node2
}

In [ ]:
# Run to re-sample
ar_mod_icar = CmdStanModel(stan_file="ar_1_hierarchical_icar.stan")
ar_mod_icar_post = ar_mod_icar.sample(data=stan_data_icar)
print(ar_mod_icar_post.diagnose())

In [ ]:
# Load model
ar_mod_icar_post = from_csv('./ar_mod_icar_post_output')
print(ar_mod_icar_post.diagnose())

### 5.1 Parameter Recovery

In [ ]:
ar_mod_icar_params = ar_mod_icar_post.stan_variables()

print("True gamma (pop intercept):", gamma_net)
print("Estimated gamma:", ar_mod_icar_params["gamma"].mean(axis=0))

print("True delta (pop coefficient):", delta_net)
print("Estimated delta:", ar_mod_icar_params["delta"].mean(axis=0))

print("True c (node intercepts):", c_net)
print("Estimated c:", ar_mod_icar_params["c"].mean(axis=0))

print("True beta (node coefficients):", beta_net)
print("Estimated beta:", ar_mod_icar_params["beta"].mean(axis=0))


In [ ]:
print(ar_mod_icar_post.summary())

## 6. Load CTA Data for Stan

In [ ]:
# Read cleaned data
cta_wide = pd.read_csv("../../data/clean/cta-data-wide.csv")
cta_adj = pd.read_csv("../../data/clean/cta-adjacency.csv")

# Extract Y matrix (station columns only)
station_cols = [c for c in cta_wide.columns if c.startswith("s_")]
Y_cta = cta_wide[station_cols].values

T_cta = Y_cta.shape[0]
K_cta = Y_cta.shape[1]

In [ ]:
# Build edge vectors from adjacency matrix
adj_ids = cta_adj["station_id"].astype(str).values
W_cta = cta_adj.drop(columns="station_id").values

node1_cta = []
node2_cta = []
for i in range(K_cta):
    for j in range(i + 1, K_cta):
        if W_cta[i, j] == 1:
            node1_cta.append(i + 1)
            node2_cta.append(j + 1)

N_edges_cta = len(node1_cta)

In [ ]:
stan_data_cta = {
    "T": T_cta,
    "K": K_cta,
    "Y": Y_cta,
    "N_edges": N_edges_cta,
    "node1": node1_cta,
    "node2": node2_cta,
}

print(N_edges_cta)

## 7. Model 3: CTA data on AR(1) Hierarchical ICAR Model

In [ ]:
# Run to re-sample
ar_mod_cta_icar = CmdStanModel(stan_file="ar_1_hierarchical_icar.stan")
ar_mod_cta_icar_post = ar_mod_cta_icar.sample(data=stan_data_cta)
print(ar_mod_cta_icar_post.diagnose())
ar_mod_cta_icar_post.save_csvfiles(dir='./ar_mod_cta_icar_output')

In [ ]:
# Load model
ar_mod_cta_icar_post = from_csv('./ar_mod_cta_icar_output')
print(ar_mod_cta_icar_post.diagnose())

In [ ]:
# Load posterior samples
draws = ar_mod_cta_icar_post.stan_variables()
draws_df = ar_mod_cta_icar_post.draws_pd()
draws_array = ar_mod_cta_icar_post.draws()

gamma = draws['gamma']
delta = draws['delta']

# posterior estimates for gamma, delta, and sigma_phi
print(f"gamma: {gamma.mean():.3f} [{np.quantile(gamma, 0.025):.3f}, {np.quantile(gamma, 0.975):.3f}]")
print(f"delta: {delta.mean():.3f} [{np.quantile(delta, 0.025):.3f}, {np.quantile(delta, 0.975):.3f}]")

In [ ]:
# Histogram for station-level betas
beta = draws['beta']     
beta_median = np.median(beta, axis=0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(beta_median, bins=30, edgecolor='black', alpha=0.75)
plt.tight_layout()
plt.show()

In [ ]:
# Histogram for station-level sigmas
sigma = draws['sigma']        
sigma_median = np.median(sigma, axis=0) 

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(sigma_median, bins=30, edgecolor='black', alpha=0.75)
plt.tight_layout()
plt.show()

In [ ]:
station_id = cta_adj["station_id"]

station_df = pd.concat([station_id, pd.Series(beta_median, name='beta_median'), pd.Series(sigma_median, name='sigma_median')], axis=1)
print(station_df)

# Look-up stations from looking at the histogram
station_select = station_df.loc[(station_df['beta_median'] < -0.325) & (station_df['beta_median'] > -0.45)]["station_id"]
print(station_select)

lookup_station(station_select, station_lookup)

In [ ]:
# Scatter plot of betas vs sigmas
sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(9, 7))

sns.scatterplot(
    data=station_df,
    x='beta_median',
    y='sigma_median',
    s=60,
    alpha=0.8,
    ax=ax,
)

ax.set_xlabel('beta')
ax.set_ylabel('sigma')

plt.tight_layout()
plt.show()

## 8. Model 4: CTA data on AR(1) Hierarchical ICAR Model with Seasonality

In [ ]:
# Seasonal data
seas_cols = [c for c in cta_wide.columns if c.startswith("mon")]
X_cta = cta_wide[seas_cols].values

P_cta = X_cta.shape[1]

In [ ]:
stan_data_cta_seas = {
    "T": T_cta,
    "K": K_cta,
    "Y": Y_cta,
    "P": P_cta,
    "X": X_cta,
    "N_edges": N_edges_cta,
    "node1": node1_cta,
    "node2": node2_cta,
}

In [ ]:
# Run to re-sample
ar_mod_cta_icar_seas = CmdStanModel(stan_file="ar_1_hier_icar_seas.stan")
mod_final = ar_mod_cta_icar_seas.sample(data=stan_data_cta_seas)
print(ar_mod_final.diagnose())
mod_final.save_csvfiles(dir='./ar_mod_cta_icar_seas_output')

In [ ]:
# Load model
ar_mod_final = from_csv('./ar_mod_cta_icar_seas_output')
print(ar_mod_final.diagnose())

In [ ]:
# Print gamma and delta posterior means

draws = ar_mod_final.stan_variables()
draws_df = ar_mod_final.draws_pd()
draws_array = ar_mod_final.draws()

gamma = draws['gamma']
delta = draws['delta']

print(f"gamma: {gamma.mean():.3f} [{np.quantile(gamma, 0.025):.3f}, {np.quantile(gamma, 0.975):.3f}]")
print(f"delta: {delta.mean():.3f} [{np.quantile(delta, 0.025):.3f}, {np.quantile(delta, 0.975):.3f}]")

In [ ]:
# Print seasonal coefficient posterior medians

seas = draws['seas']
print("Posterior medians:")
for i, m in enumerate(np.median(seas, axis=0)):
    print(f"  Month {i+2}: {m:+.3f}")  # Feb=2 through Dec=12 if January is baseline

In [ ]:
beta = draws['beta']          
beta_median = np.median(beta, axis=0) 

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(beta_median, bins=30, edgecolor='black', alpha=0.75)

plt.tight_layout()
plt.show()

In [ ]:
sigma = draws['sigma']               
sigma_median = np.median(sigma, axis=0)  

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(sigma_median, bins=30, edgecolor='black', alpha=0.75)

plt.tight_layout()
plt.show()

In [ ]:
station_df = pd.concat([station_id, pd.Series(beta_median, name='beta_median'), pd.Series(sigma_median, name='sigma_median')], axis=1)
print(station_df)

station_select = station_df.loc[(station_df['beta_median'] < -.1) & (station_df['beta_median'] > -.2)]["station_id"]
print(station_select)

lookup_station(station_select, station_lookup)

In [ ]:
# Scatter plot of betas vs sigmas
sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(9, 7))

sns.scatterplot(
    data=station_df,
    x='beta_median',
    y='sigma_median',
    s=60,
    alpha=0.8,
    ax=ax,
)

ax.set_xlabel('beta')
ax.set_ylabel('sigma')

plt.tight_layout()
plt.show()

## 9. Posterior Predictive Checks

In [ ]:
# Select stations for PPCs
station_select = ['Sox-35th-Dan Ryan', 'Clark/Lake', 'Loyola', 'LaSalle']
station_select_id = lookup_station(station_select, station_lookup)

check_idx = [
    np.where(cta_adj['station_id'].values == sid)[0][0] + 1
    for sid in station_select_id
]

print(check_idx)

In [ ]:
# Generate quantities data and find the predicted values
gq_data = {
    "T": T_cta,
    "K": K_cta,
    "Y": Y_cta,
    "P": P_cta,
    "X": X_cta,
    "N_edges": N_edges_cta,
    "node1": node1_cta,
    "node2": node2_cta,
    "N_check": len(check_idx),
    "check_idx": check_idx
}

gq_model = CmdStanModel(stan_file='generate_quantities.stan')
gq_fit = gq_model.generate_quantities(data=gq_data, previous_fit=ar_mod_final)
y_rep = gq_fit.stan_variables()['y_rep']

In [ ]:
# Build data frame to export to R
ppc_records = []
for i, station in enumerate(station_select):
    k = check_idx[i] - 1
    median = np.median(y_rep[:, :, i], axis=0)
    lower = np.quantile(y_rep[:, :, i], 0.025, axis=0)
    upper = np.quantile(y_rep[:, :, i], 0.975, axis=0)
    observed = Y_cta[:, k]
    
    for t in range(T_cta):
        ppc_records.append({
            'station': station,
            'week': t,
            'observed': observed[t],
            'median': median[t],
            'lower': lower[t],
            'upper': upper[t],
        })

ppc_df = pd.DataFrame(ppc_records)
ppc_df.to_csv('../../data/clean/ppc_summary.csv', index=False)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
axes = axes.flatten()

handles, labels = None, None

for i in range(len(station_select)):
    ax = axes[i]
    k = check_idx[i] - 1

    median = np.median(y_rep[:, :, i], axis=0)
    lower = np.quantile(y_rep[:, :, i], 0.025, axis=0)
    upper = np.quantile(y_rep[:, :, i], 0.975, axis=0)

    ax.fill_between(range(T_cta), lower, upper, alpha=0.3, color='steelblue', label='95% PPC interval')
    ax.plot(median, color='steelblue', linewidth=1, alpha=0.8, label='Posterior median')
    ax.plot(Y_cta[:, k], color='black', linewidth=0.8, label='Observed')

    ax.axhline(0, color='gray', linestyle=':', alpha=0.4)
    ax.set_title(station_select[i])
    ax.set_ylabel(r'$\Delta \log y$')

    if handles is None:
        handles, labels = ax.get_legend_handles_labels()

axes[2].set_xlabel('Week')
axes[3].set_xlabel('Week')

fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False)

fig.suptitle('Posterior predictive checks', y=1.06, fontsize=13)
plt.tight_layout()
plt.show()